# Short-possession recovery: contact history, spell weighting and adaptive switching

The precommitted F1_TARGET.md protocol fixes four graph/loss variants and two
decoders. Reuse existing graph/natural fits; newly train the other three variants
with five seed-0 whole-window folds and three final seeds, 60 epochs each.
Select the variant using training OOF only, save the choice, then score only that
new selection on the unchanged draft validation. No manual labels or score edits.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_recovery')
WORK.mkdir(parents=True,exist_ok=True)
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'possession_recovery/experiment.py').is_file()]
assert len(roots)==1
CODE=roots[0]
configs=list(INPUT.rglob('recovery_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
labels=list(INPUT.rglob('relational_input.json'));assert len(labels)==1
LABELS=labels[0].parent
base=[p for p in INPUT.rglob('features.npz') if sha(p)==plan['feature_sha256']]
audio=[p for p in INPUT.rglob('audio_cache.npz') if sha(p)==plan['audio_sha256']]
assert len(base)==len(audio)==1
FEATURES,AUDIO,GRAPH=base[0],audio[0],FILES/'history_graph.npz'
assert sha(GRAPH)==plan['history_graph_sha256']
assert sha(CODE/'F1_TARGET.md')==plan['protocol_sha256']
for split in ('train','validation'):assert sha(LABELS/f'{split}.csv')==plan[f'{split}_sha256']
sources=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='team_symmetric_graph_memory_span_ablation']
assert len(sources)==1
SOURCE=sources[0]
DEPS=Path('/tmp/barca_recovery_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',
    HF_HUB_OFFLINE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full recovery suite passed; starting precommitted real study.',flush=True)


In [ ]:
RUN=WORK/'study'
run('-m','possession_recovery','compare','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--source-study',SOURCE,'--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv',
    '--out',RUN,'--epochs','60','--seeds','0','1','2','--device','cuda','--bootstrap','1000')


In [ ]:
output=RUN/'selected_regenerated.csv'
run('-m','possession_recovery','predict','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--model',RUN/'deployment/model','--out',output,'--device','cuda')
assert output.read_bytes()==(RUN/'selected_predictions.csv').read_bytes()
report=json.loads((RUN/'report.json').read_text())
source={str(p.relative_to(CODE)):sha(p) for root in ('possession_recovery','relational_possession','temporal_possession','learned_possession','structured_possession') for p in (CODE/root).glob('*.py')}
protocol=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
    source_sha256=source,private_inputs=plan,git_checkpoint=plan['git_checkpoint'],
    selected_output_regenerated=True,prediction_sha256=sha(output),
    references_unchanged=all(sha(LABELS/f'{split}.csv')==plan[f'{split}_sha256'] for split in ('train','validation')))
assert protocol['references_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('Byte-identical selected recovery regeneration.',flush=True)
print('RECOVERY GPU RUN COMPLETE',flush=True)
